### Step 1: Merge the LoRA Adapter with the Base Model

In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

model_id = "Qwen/Qwen3.5-4B-Base"
adapter_model_dir = "./qwen3.5_lora_adapter"
merged_output_dir = "./qwen3.5_merged"

print("Loading tokenizer and base model...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto"  # Safe for memory; change to "auto" if you have plenty of VRAM
)

print("Merging weights... (this might take a minute)")
model = PeftModel.from_pretrained(base_model, adapter_model_dir)
merged_model = model.merge_and_unload()  # Fuses adapter layers into base layers

print(f"Saving merged model to {merged_output_dir}...")
merged_model.save_pretrained(merged_output_dir)
tokenizer.save_pretrained(merged_output_dir)
print("Merge complete!")

/home/eternalcm/RandD/yolo-autoresearch/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading tokenizer and base model...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Fetching 2 files: 100%|██████████| 2/2 [00:00<00:00, 4076.10it/s]
[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
Loading weights: 100%|██████████| 426/426 [00:00<00:00, 736.73it/s]


Merging weights... (this might take a minute)
Saving merged model to ./qwen3.5_merged...


Writing model shards: 100%|██████████| 1/1 [00:04<00:00,  4.18s/it]

Merge complete!


### Step 2: Convert the Merged Model to GGUF Format

In [ ]:
# 1. Clone the llama.cpp repository
!git clone https://github.com/ggml-org/llama.cpp.git
!cd llama.cpp

Cloning into 'llama.cpp'...
remote: Enumerating objects: 102760, done.
remote: Counting objects: 100% (85/85), done.
remote: Compressing objects: 100% (66/66), done.
remote: Total 102760 (delta 24), reused 20 (delta 19), pack-reused 102675 (from 2)
Receiving objects: 100% (102760/102760), 406.94 MiB | 1.39 MiB/s, done.
Resolving deltas: 100% (72058/72058), done.


#### 2. Install the necessary Python dependencies for conversion
```bash
uv add -r requirements.txt
```

**Note on Quantization**: Using `--outtype q8_0` compresses the model to **8-bit** integer weights.

This shrinks the file size dramatically while maintaining almost **99%** of the unquantized model's accuracy. 

If you want raw, uncompressed precision, change it to `--outtype bf16`.

#### 3. Convert your merged model to a compressed 8-bit GGUF file
```bash
python convert_hf_to_gguf.py ../qwen3.5_merged --outfile ../qwen3.5_q8.gguf --outtype q8_0
```

### Step 3: Initialize and Create Your Ollama Model
Now that you have a `qwen3.5_q8.gguf` file, you need to tell Ollama how to chat with it.

1. Go back to your main working directory (where your `.gguf` file is located).

2. Create a new text file named `Modelfile` (no file extension) and paste the following configuration inside it:
```dockerfile
# Point to your newly generated GGUF file
FROM ./qwen3.5_q8.gguf

# Set Qwen's native ChatML template format so it understands system/user prompt tags
TEMPLATE """{{ if .System }}<|im_start|>system
{{ .System }}<|im_end|>
{{ end }}{{ if .Prompt }}<|im_start|>user
{{ .Prompt }}<|im_end|>
{{ end }}<|im_start|>assistant
"""

# Set your default runtime parameters
PARAMETER temperature 0.7
PARAMETER top_p 0.9
PARAMETER stop "<|im_start|>"
PARAMETER stop "<|im_end|>"

# Optional: Give your model a default personality or instructions
SYSTEM """You are a helpful AI assistant fine-tuned for a specialized task."""
```
Save the `Modelfile`. Now, run the following terminal commands to build and start using your model locally:

```bash
# Compile the model into Ollama
ollama create specialized-qwen -f Modelfile

# Run your model!
ollama run specialized-qwen
```

# ❌ Error

**Error: 500 Internal Server Error**: llama-server process has terminated: exit status 1: 

`error loading model: missing tensor 'blk.32.attn_norm.weight'`

## Reason

This is a known quirk specific to the **Qwen 3.5** architecture when converting fine-tuned or merged models to GGUF.

#### Why Is This Happening?
**Qwen 3.5** features a hybrid design that includes an auxiliary **MTP (Multi-Token Prediction)** block used for speculative decoding. A standard 4B model has **32 core transformer layers (indexed 0 to 31)** plus this **1 extra MTP layer**.

When you merged your LoRA adapter, the script dropped the MTP layer (which is normal and safe, as it isn't required for regular text generation). However, the model's configuration file still told the `convert_hf_to_gguf.py` script that the MTP layer existed. As a result, the converter wrote `block_count = 33` into the GGUF file's metadata header.

When Ollama attempts to load the model, it reads `33 blocks`, looks for the 33rd layer (index `32`), fails to find `blk.32.attn_norm.weight`, and crashes.

## How to Fix It
Since a 4B model is lightweight, the easiest and most reliable solution is to fix the model's configuration and re-run the GGUF conversion.

### Step 1: Clean Up the Configuration
1. Open your merged model directory folder: `./qwen3.5_merged`
2. Open the `config.json` file in a text editor.
3. Look for the line that mentions `mtp` (usually `"mtp_num_hidden_layers": 1`).
4. Change its value from `1` to `0`, or delete that line entirely. Save and close the file.
### Step 2: Re-convert the Model
Go back to your terminal inside the `llama.cpp` folder and run the conversion script again. If you have a recent clone of `llama.cpp`, you can also explicitly pass the `--no-mtp` flag as an extra safety measure:

```bash
python convert_hf_to_gguf.py ../qwen3.5_merged --outfile ../qwen3.5_q8.gguf --outtype q8_0 --no-mtp
```
### Step 3: Force Ollama to Rebuild
Because Ollama caches model layers, you need to delete the broken instance and rebuild it fresh using your existing `Modelfile`:
```bash
# 1. Remove the broken model instance
ollama rm specialized-qwen

# 2. Re-compile the corrected GGUF file
ollama create specialized-qwen -f Modelfile

# 3. Launch it again
ollama run specialized-qwen
```
Once rebuilt with the correct header metadata stating a `block_count` of 32, Ollama will load your fine-tuned model cleanly without looking for a phantom 33rd layer.